# Python in the Smoke 💨 🐍 💨
## Notebook i: Exploring data

---

**Repo:** [`python-in-the-smoke`](https://github.com/MarkGotham/python-in-the-smoke)

**By:** Mark Gotham

**Licence:** MIT

**Colour and symbol key:**
- 🟢 A block of information.
- 🟣 A core exercise.
- ⭐ An optional exercise.
- ✅ The solution to the exercise above it.
- 📝 A note with important information.
- ⚠️ Common errors, and how to fix them.
- `Grey text like this` shows an exact code or command. Type it exactly as shown, including any capital letters, spaces, and punctuation.

**Assumed knowledge:**

This notebook builds on notebooks a to h, and continues directly from Notebook h (cleaning data).
Each idea in the table below has appeared before,
and now returns in the form that's directly relevant for exploring data.

| a-h | Assumed knowledge                 | How it appears here                                                          |
|-----|-----------------------------------|------------------------------------------------------------------------------|
| c   | Booleans, comparisons             | Filters such as `price_gbp == 0`, and tests such as `dayofweek >= 5`          |
| d   | the accumulator pattern           | Counting and totalling, which pandas does in one step (as in Notebook f)     |
| f   | selecting, sorting, summarising   | `.sort_values`, `.mean()`, `.sum()`, filtering, new columns, `.str` methods  |
| g   | `read_csv`, `Path`                | Loading the cleaned file, and checking that it exists                        |
| h   | missing values, `parse_dates`     | `NaN` and `NaT` in every summary; `dropna(subset=...)`; `.copy()`            |

**Learning Objectives**: After completing this notebook, you will be able to ...
- Turn a cultural question into a filter or a summary
- Count categories, stating the denominator and the missing values
- Compare groups using `groupby` and `.agg`, keeping track of missing group labels
- Calculate occupancy rates, and distinguish mean event occupancy from the share of all available places filled
- Use dates to compare groups of events, without losing events with unknown dates by accident
- Write conclusions that match the data available

---
## 🟢 Information: Reminders

Run each cell as you go (▶ or Shift+Enter).
Complete the core exercises, and try the optional exercises if you have time.
Solutions appear below each exercise.
Imports and variables do not carry over between notebooks.

The programme we explore is the **fictional** one from Notebook h.
Event names, dates, prices and attendance figures are invented for practice.
It includes deliberate data problems (which Notebook h dealt with, or left missing on purpose).
These are not claims about real events or boroughs.

---
## 🟢 Information: Load the cleaned programme

This notebook loads the file that Notebook h saved: `data/practice_data_clean.csv`.
First, check that it's there.
This should give `True`.
If it gives `False`, run Notebook h up to the 'Save and document' section, then come back.

In [ ]:
from pathlib import Path
import pandas as pd

Path("data/practice_data_clean.csv").exists()

Now load it.
We use `parse_dates=["date"]` so that `date` comes back as dates, rather than text (as at the end of Notebook h).

In [ ]:
events = pd.read_csv("data/practice_data_clean.csv", parse_dates=["date"])
events.head()

In [ ]:
events.shape

In [ ]:
events.isna().sum()

Remember what we are (and are not) exploring.
This is **one fictional programme**, not a measure of cultural provision across London.
More events in this table does not mean better access to culture in a borough.

The four `1`s in the counts above are the problems that Notebook h left unresolved, on purpose:
one missing `borough`, `attendance`, `price_gbp` and `date`.
Keep them in mind: every summary in this notebook has to decide what to do about them.

---
## 🟢 Information: Start with a question

Good exploration starts with a question that's a suitable balance of:
- **interesting** (otherwise, why bother?), and
- **feasible** (small and clear enough for the data at hand to answer).

Some potentially good examples:
- Which art forms occur most in this programme?
- How many events are free?
- How full were events with recorded attendance?
- Were weekend events fuller than weekday events?

... and some are probably best avoided:
- Which borough has the best culture? (What does "best" mean?)
- Is London more culturally active than Paris? (Feasible in principle, but we would need data for Paris.)

---
### 🟣 Core Exercise: Count the art forms

1. Use `.value_counts()` to count the events for each art form.
2. Count how many events have a known price of exactly zero.

---
---
---

### ✅ Solution: Count the art forms

In [ ]:
events["artform"].value_counts()

In [ ]:
(events["price_gbp"] == 0).sum()

Each of the five art forms has three events.

Three events have a recorded price of £0.
(A missing price is not equal to zero, so the event with an unknown price isn't counted here.)

---
## *** 🟢 Information: Counts and denominators

- `value_counts()` leaves out missing values by default; `dropna=False` includes them.
- When turning counts into percentages, always be clear about the **denominator**: "a percentage of what?"

Here, `.fillna("Not recorded")` gives the unknown borough a temporary label, so that it shows up in the counts.
It's a *display label*: the data in `events` doesn't change (Notebook h).

In [ ]:
borough_counts = events["borough"].fillna("Not recorded").value_counts()
borough_counts

In [ ]:
borough_percent = (100 * borough_counts / len(events)).round(1)
borough_percent

---
### 🟣 Core Exercise: Free as a share of what?

Calculate the percentage of **known-price events** that are free.
- Use `.notna().sum()` for the denominator.
- Why is this different from dividing by all events?

---
---
---

### ✅ Solution: Free as a share of what?

In [ ]:
known_price_count = events["price_gbp"].notna().sum()
free_count = (events["price_gbp"] == 0).sum()
round(100 * free_count / known_price_count, 1)

21.4% of the 14 known-price events are free.
- Dividing by all 15 gives 20%: the share of all listed events that are *known* to be free.
- Neither tells us whether the unknown-price event was free.
- If no prices were known, the first percentage would be undefined.

Both numbers are honest, as long as you say which one you're giving.

---
## 🟢 Information: Numerical summaries

Some useful numerical tools in pandas:
- `.describe()` gives the count, average, spread and quartiles for numerical columns (as in the optional exercise in Notebook f).
- `.mean()` is the average. `.median()` is the middle value, which is less affected by extreme values.
- `.count()` gives the number of values that are **not** missing.

Missing values are skipped by most summaries.
So always read the `count`: price, capacity and attendance can describe different sets of events.

Don't summarise identifiers as numbers.
(`event_id` is text, so `.describe()` will rightly leave it out.)

In [ ]:
events[["price_gbp", "capacity", "attendance"]].describe()

---
### 🟣 Core Exercise: Typical attendance

- Find the mean, the median and the number of recorded `attendance` values.
- Round the mean to one decimal place.
- Which is the better description of a "typical" event?

---
---
---

### ✅ Solution: Typical attendance

In [ ]:
events["attendance"].mean().round(1)

In [ ]:
events["attendance"].median()

In [ ]:
events["attendance"].count()

Mean: 85.5. Median: 79. Count: 14.
- The missing attendance is excluded, not treated as zero.
- "Typical" needs a definition: the mean and the median answer different questions.
Here they are fairly close, so either is reasonable; with a few very large events, they would be further apart.

---
## 🟢 Information: Compare groups with `groupby`

To compare groups (for example, art forms), we **split** the table into groups, **summarise** each one, and **combine** the results.
`groupby` does all three:
- `groupby("artform")` splits the table into one group per art form.
- A summary after it (such as `.mean()`) then runs within each group.

Two counts can easily be confused:
- `.size()` counts the **rows** in each group.
- `.count()` counts the **non-missing values** in the selected column.

Compare them before interpreting any group average.

In [ ]:
events.groupby("artform").size()

In [ ]:
events.groupby("artform")["attendance"].count()

In [ ]:
events.groupby("artform")["attendance"].mean().round(1)

`dance` has three events, but only two with recorded attendance, so its mean is based on two events.

`.agg` (short for "aggregate") gives several summaries at once.
You pass it a list of the summaries you want, as text:

In [ ]:
events.groupby("artform")["attendance"].agg(["count", "mean"])

---
### 🟣 Core Exercise: Several summaries together

Build a summary table of `attendance` by art form with four columns: `count`, `mean`, `median` and `sum`.
Then sort it by the mean, largest first (`sort_values`, as in Notebook f), and round to one decimal place.

Which art form has incomplete attendance data?

---
---
---

### ✅ Solution: Several summaries together

In [ ]:
artform_summary = events.groupby("artform")["attendance"].agg(["count", "mean", "median", "sum"])
artform_summary.sort_values("mean", ascending=False).round(1)

`dance` has attendance for only two of its three events.
- A `sum` is the total of the *recorded* attendance, not necessarily the true total.
- All groups here have at least one recorded value.
Without any, the sum would show `0`, which would be misleading.

---
## *** 🟢 Information: Missing group labels

There are two different ways a row can disappear from a summary:
- a missing **measurement**, such as attendance, is skipped by `.count()` and `.mean()`;
- a missing **group label**, such as borough, is left out by `groupby` itself, unless we say otherwise.

**Predict, then run:** there are 15 events, and one has no recorded borough.
Will the counts below add up to 14 or 15?

In [ ]:
grouped_boroughs = events.groupby("borough").size()
print(grouped_boroughs)
print(f"Events included: {grouped_boroughs.sum()}")

Only 14 events are included: `groupby` has left out the event with no borough.
Add `dropna=False` to keep it in a separate group, just as we did with `value_counts` in Notebook h:

In [ ]:
grouped_boroughs = events.groupby("borough", dropna=False).size()
print(grouped_boroughs)
print(f"Events included: {grouped_boroughs.sum()}")

Now all 15 events are included. The missing label marks an unknown borough; it is not a new borough.
We have kept the event without inventing its location or changing `events`.

`dropna=False` keeps missing **group labels**. It does not make a missing attendance value known:
`.count()` and `.mean()` still skip missing measurements within each group.

---
## 🟢 Information: Attendance relative to capacity

An audience of 100 would nearly fill a 110-place venue, but leave most of a 500-place venue empty.
**Occupancy** expresses attendance as a percentage of the available places:

`occupancy_pct = 100 * attendance / capacity`

Before dividing, check that capacities are known and greater than zero.
(These checks appeared in an optional exercise in Notebook h; run them here to make sure, even if you skipped that exercise.)

In [ ]:
print(f"Missing capacities: {events['capacity'].isna().sum()}")
print(f"Zero or negative capacities: {(events['capacity'] <= 0).sum()}")

Both counts should be `0` for this file, so every capacity is suitable for the calculation.
If either count is non-zero in another dataset, investigate those records before calculating rates.

We can now add a percentage for each event.
Where attendance is unknown, occupancy remains unknown too.

In [ ]:
events["occupancy_pct"] = 100 * events["attendance"] / events["capacity"]
events[["event", "capacity", "attendance", "occupancy_pct"]]

---
### 🟣 Core Exercise: The fullest events

1. Display the three events with the highest recorded occupancy.
Include the event name and the percentage, rounded to one decimal place.
Exclude missing attendance explicitly (`dropna(subset=...)`).
2. Is third place clear-cut?
Check by filtering for events with `occupancy_pct` of 90 or more.

---
---
---

### ✅ Solution: The fullest events

In [ ]:
known = events.dropna(subset=["attendance"])
known.sort_values("occupancy_pct", ascending=False)[["event", "occupancy_pct"]].head(3).round(1)

In [ ]:
events[events["occupancy_pct"] >= 90][["event", "occupancy_pct"]]

- Summer Chorus (95.0%) and Small Stage (92.9%) are clear.
- **Third place is a tie**: `Paper Stories` and `Southbank Sketches` are both exactly 90.0%.
`.head(3)` shows only one of them, silently, because it simply cuts the sorted table after three rows.
When you report a "top N", consider checking the next row/s too and/or
having a secondary sort option to control sort order in the case of a tie.
- Remember: a high occupancy rate doesn't mean a large audience.
`Paper Stories` is full-ish with 54 people.

---
## *** 🟢 Information: Two ways to combine occupancy rates

Suppose a **fictional London arts centre** holds two sessions:
a small drawing workshop and a larger evening talk.
All the figures below are invented, and both attendance counts are known.

In [ ]:
sessions = pd.DataFrame(
    {
        "session": ["Drawing workshop", "Evening talk"],
        "capacity": [20, 180],
        "attendance": [20, 90],
    }
)
sessions["occupancy_pct"] = 100 * sessions["attendance"] / sessions["capacity"]
sessions

The workshop is 100% full; the talk is 50% full.
**Before running the next cells:** is the share of all available places filled halfway between those rates?
Think about how many places each session contributes.

First, take the **mean session occupancy**:

In [ ]:
mean_session_occupancy = sessions["occupancy_pct"].mean()
print(f"Mean session occupancy: {round(mean_session_occupancy, 1)}%")

Now combine the attendance counts and capacities before dividing:

In [ ]:
overall_session_occupancy = 100 * sessions["attendance"].sum() / sessions["capacity"].sum()
print(f"Share of all available places filled: {round(overall_session_occupancy, 1)}%")

| Question | Calculation | Result |
|----------|-------------|--------|
| What was the mean occupancy of a session? | `(100 + 50) / 2` | 75% |
| What share of all available places was filled? | `100 * (20 + 90) / (20 + 180)` | 55% |

The first gives **each session equal weight**: the 20-place workshop counts as much as the 180-place talk.
The second gives **each available place equal weight**, so the larger session contributes more.
Both are useful; choose the one that answers your question, and name it clearly.

---
### 🟣 Core Exercise: Occupancy for the programme

Return to `events`, the programme from Notebook h.
1. Make `known_occupancy` with only the rows that have both `attendance` and `capacity`.
   Use `dropna(subset=[...])`, and count the remaining events.
2. Calculate the mean of their `occupancy_pct` values.
3. Calculate `100 * total attendance / total capacity` using **that same table** for both totals.
   Round both rates to one decimal place.
4. Which result answers "what share of the available places was filled across these events"?
   Why must we leave the unknown-attendance event's capacity out of this calculation too?

---
---
---

### ✅ Solution: Occupancy for the programme

In [ ]:
known_occupancy = events.dropna(subset=["attendance", "capacity"])
mean_event_occupancy = known_occupancy["occupancy_pct"].mean()
overall_occupancy = 100 * known_occupancy["attendance"].sum() / known_occupancy["capacity"].sum()

print(f"Events with both values: {len(known_occupancy)}")
print(f"Mean event occupancy: {round(mean_event_occupancy, 1)}%")
print(f"Share of available places filled: {round(overall_occupancy, 1)}%")

There are 14 events with both values.
Their mean event occupancy is **85.0%**; the share of available places filled is **84.9%**
(1,197 recorded attendances across 1,410 available places).
The two answers happen to be close in this programme; the arts-centre example shows why they need not be.

The second result answers the question about available places.
Including the unknown-attendance event's capacity would count its places without counting its audience,
effectively treating that unknown audience as zero.
Neither result describes the occupancy of all 15 events: one attendance is still unknown.

---
## *** 🟢 Information: Working with dates

In Notebook h, we made `date` a proper date column, and `parse_dates` kept it that way.
So now pandas can pull the parts out of it.
Just as `.str` holds the tools for text (Notebook f), `.dt` holds the tools for dates:
- `.dt.day_name()` gives the name of the weekday, such as `Saturday`.
- `.dt.dayofweek` gives the weekday as a number: Monday is `0`, and Sunday is `6`.

In [ ]:
events["weekday"] = events["date"].dt.day_name()
events[["event", "date", "weekday"]]

The event with no date has no weekday either (`NaN`).

How many events took place at the weekend (Saturday or Sunday)?
`.isin([...])` tests whether each value is in a list:

In [ ]:
events["weekday"].isin(["Saturday", "Sunday"]).sum()

⚠️ **A silent trap.**
Five events are at the weekend, so it might seem that `15 - 5 = 10` events were on weekdays.
But only 14 events have a date, so only `14 - 5 = 9` events are *known* to be on weekdays.
The event with no date isn't a weekday event: it's an unknown-day event.

A test such as "is this a weekend?" gives `False` for a missing date,
so an event with no date can quietly slip into the "weekday" group.
(The same thing happened with `attendance > capacity` in Notebook h.)
The fix is to select the events with a known date *first*.

---
### 🟣 Core Exercise: Weekend or weekday?

Were weekend events fuller than weekday events?
Use **mean event occupancy**, so that each event has equal weight within its group.
1. Make a table called `dated` with only the events that have **both** a known `date` and a known `attendance`.
Add `.copy()` (as in Notebook h).
How many events are in it?
2. Add a column `is_weekend` to `dated`: `True` for Saturday and Sunday, using `dayofweek` and a comparison with `5`.
3. Group by `is_weekend` and find the `count` and `mean` of `occupancy_pct`, rounded to one decimal place.
4. Write down one finding and one limitation.
Say whether your comparison weights events or available places equally.

---
---
---

### ✅ Solution: Weekend or weekday?

In [ ]:
dated = events.dropna(subset=["date", "attendance"]).copy()
dated.shape

In [ ]:
dated["is_weekend"] = dated["date"].dt.dayofweek >= 5

In [ ]:
dated.groupby("is_weekend")["occupancy_pct"].agg(["count", "mean"]).round(1)

13 events have both a date and an attendance.
- **Finding:** weekday events averaged 86.5% occupancy (9 events), and weekend events 82.9% (4 events).
In this table, weekend events had slightly lower mean occupancy.
Each event has equal weight within its group; these are not percentages of all available places filled.
- **Limitations:** there are only four weekend events, and all of the figures are invented.
A gap of a few percentage points between such small groups says very little,
and it wouldn't show that the day of the week *causes* the difference.

For comparison: if we had skipped step 1 and just grouped everything,
the undated event would have been counted as a weekday, and we'd have got 85.8% for "weekdays" instead of 86.5%.
The error is small here, but it is silent.
Its size depends on how many dates are missing and the occupancy of those events.

---
### 🟣 Core Exercise: Explore a borough

Choose one recorded borough.
- Filter its events, count them, and find the total of their known attendance.
- Write one finding and one limitation.

---
---
---

### ✅ Solution: Explore a borough

In [ ]:
hackney = events[events["borough"] == "Hackney"]
print(len(hackney))
print(hackney["attendance"].count())
print(hackney["attendance"].sum())

Hackney has three listed events, all with recorded attendance, totalling 355.
(We printed the `count` as well: if it had been lower than the number of events, the `sum` would be incomplete.)

Some limitations of this data:
- The table is fictional: it is not a census of Hackney's culture.
- Attendance counts *visits* to events, not unique people: the same person can attend several.
- The event with a missing borough might be a Hackney event: we don't know.

---
## 🟢 Information: Questions the data cannot settle

Price and attendance can be explored together, but a pattern wouldn't prove that price *caused* attendance.
As they say, correlation does not imply causation!
- Capacity, art form, publicity and the day of the week might matter too.
- There are only 15 invented events.

Likewise:
- A date identifies a scheduled event; a missing attendance is not evidence that an event was cancelled.
- A missing borough is not a new place: it's a gap.

---
### 🟣 Core Exercise: A short findings note

Write three sentences in a Markdown cell:
1. a numerical finding, with its denominator,
2. a comparison,
3. a limitation.

Include the words "fictional programme".

Double-click this cell to edit it.

---
---
---

### ✅ Solution: A short findings note

**Example:**
In this fictional programme, 3 of the 14 known-price events are free (21.4%).
Music has the highest mean recorded attendance, at 143, across three events.
These invented figures describe only the supplied programme, and they can't establish borough-wide provision, or the effect of price or day of the week on attendance.

---
### ⭐ Optional Exercise: Real data: venues per borough

Apply the same ideas to the real music venues from Notebooks g and h.
1. Load `data/Music_venues_grassroots.csv` as `music`.
2. Standardise `borough_name` with `.str.strip().str.title()`, and label the missing ones `"Not recorded"`.
3. Count the venues per borough, and show the top five, as percentages of **all 100 venues** (rounded to one decimal place).
4. What does the denominator mean for the "Not recorded" group?

---
---
---

### ✅ Solution: Real data: venues per borough

In [ ]:
music = pd.read_csv("data/Music_venues_grassroots.csv")
boroughs = music["borough_name"].str.strip().str.title().fillna("Not recorded")
venue_counts = boroughs.value_counts()
(100 * venue_counts / len(music)).round(1).head(5)

- Hackney and Islington tie at 15% each, then **"Not recorded" and Camden** at 14%.
- "Not recorded" is almost as large as the biggest recorded boroughs (14% versus 15%).
So a claim such as "Islington has the most venues" would be shaky:
it ties with Hackney, and the 14 unrecorded venues could change the picture.
- Also remember that this is a snapshot from 2019, so it's not a current list of venues.

---
## 🟢 Information: Summary

| Question                              | Useful tool                                                   |
|---------------------------------------|---------------------------------------------------------------|
| How many in each category?            | `.value_counts(dropna=False)`                                 |
| How many recorded numbers?            | `.count()`                                                    |
| What is typical?                      | `.mean()`, `.median()`                                        |
| How do groups differ?                 | `.groupby("artform")`                                         |
| Keep a missing group label?           | `.groupby("borough", dropna=False)`                            |
| Several group summaries?              | `.groupby(...)[col].agg(["count", "mean", "median"])`         |
| Rows in each group?                   | `.groupby(...).size()`                                        |
| Attendance relative to capacity?      | `100 * attendance / capacity`                                 |
| Mean event occupancy?                 | `known_occupancy["occupancy_pct"].mean()`                       |
| Share of available places filled?     | `100 * known_occupancy["attendance"].sum() / known_occupancy["capacity"].sum()` |
| Which day of the week?                | `df["date"].dt.day_name()`, `df["date"].dt.dayofweek`         |
| Rows with a known value (and a date)? | `df.dropna(subset=["attendance", "date"]).copy()`             |

---
## ⚠️ Common Errors

- **Using `.count()` when you mean the number of rows.**
`.count()` skips missing values.
Use `.size()` for the number of rows in a group, or `len(df)` for the whole table.
- **Reporting a percentage without a denominator.**
Say whether unknown values were excluded.
- **Losing records with unknown group labels.**
`groupby` leaves these out by default. Use `dropna=False` when the unknown group belongs in the summary.
- **Confusing mean event occupancy with overall occupancy.**
A mean of rates gives each event equal weight; a ratio of totals gives each available place equal weight.
Name the measure you report.
- **Dividing total known attendance by the capacity of *all* events.**
Use the same rows in the numerator and the denominator.
- **Letting an unknown slip into a group.**
A test such as `dayofweek >= 5` is `False` for a missing date, so the event joins the "weekday" group.
Select the rows with known values first.
- **Trusting `.head(3)` for a "top three".**
It cuts the table after three rows, even if the next row ties with the third.
- **Calling a recorded attendance total "unique visitors".**
One person can attend several events.
- **`FileNotFoundError` on loading.**
Notebook h creates the file this notebook loads.
Run Notebook h first, and check that `data/practice_data_clean.csv` exists.

End

---